# Stage 3: CLV Target Rebuild
The source data's customer_lifetime_value column had max correlation below 0.05 with every feature. This stage replaces it with a deterministic, learnable value derived from business logic, then re-splits the features file.

CLV = annual_premium x expected_years_retained x profit_margin_proxy

Input: s3://ins-churn-data/cleaned/fact_policy_activity_cleaned.csv, s3://ins-churn-data/features/features.csv

Output: updated features/features.csv, train.csv, val.csv

In [ ]:
import boto3, pandas as pd, numpy as np, io, os, json, logging
from sklearn.model_selection import train_test_split

logging.basicConfig(level=logging.INFO, format='%(levelname)s | %(message)s')
log = logging.getLogger(__name__)

BUCKET_NAME = 'ins-churn-data'
REGION = 'us-east-1'
s3 = boto3.client('s3', region_name=REGION)


def read_csv_from_s3(bucket, key, **kwargs):
    response = s3.get_object(Bucket=bucket, Key=key)
    return pd.read_csv(io.BytesIO(response['Body'].read()), **kwargs)


def write_csv_to_s3(df, bucket, key):
    buffer = io.StringIO()
    df.to_csv(buffer, index=False)
    s3.put_object(Bucket=bucket, Key=key, Body=buffer.getvalue())
    log.info('  saved to s3://%s/%s (%d rows)', bucket, key, len(df))


log.info('Imports ready.')

In [ ]:
fact_raw = read_csv_from_s3(BUCKET_NAME, 'cleaned/fact_policy_activity_cleaned.csv')

# Annualised premium, core revenue driver
fact_raw['annual_premium'] = fact_raw['premium_amount'] * (
    12 / fact_raw['policy_tenure_months'].clip(lower=1)
)

# Churn risk composite from columns that exist and have meaning
fact_raw['churn_risk'] = (
    (fact_raw['payment_delay_days'].fillna(0) / 30) +
    (fact_raw['claim_frequency'].fillna(0) * 0.5) +
    (fact_raw['customer_complaints'].fillna(0) * 0.3)
).clip(lower=0)

fact_raw['retention_prob'] = (1 / (1 + fact_raw['churn_risk'])).clip(0.2, 0.95)
fact_raw['expected_years'] = (
    fact_raw['retention_prob'] / (1 - fact_raw['retention_prob'])
).clip(upper=10)

profit_margin = (
    (fact_raw['transaction_amount'] - fact_raw['premium_amount'].fillna(0)) /
    fact_raw['transaction_amount'].replace(0, np.nan)
).clip(0.05, 0.95).fillna(0.3)

fact_raw['clv_rebuilt'] = (
    fact_raw['annual_premium'] * fact_raw['expected_years'] * profit_margin
).round(2)

print(fact_raw['clv_rebuilt'].describe().round(2))

In [ ]:
# Replace CLV in features CSV and re-split
features = read_csv_from_s3(BUCKET_NAME, 'features/features.csv')

assert len(features) == len(fact_raw), (
    f'Row count mismatch: features has {len(features)} rows, '
    f'fact_policy_activity_cleaned has {len(fact_raw)} rows. '
    f'Row order must be aligned before assigning clv_rebuilt by position.'
)

features['customer_lifetime_value'] = fact_raw['clv_rebuilt'].values
write_csv_to_s3(features, BUCKET_NAME, 'features/features.csv')

df_train, df_val = train_test_split(features, test_size=0.2, random_state=42)
write_csv_to_s3(df_train, BUCKET_NAME, 'features/train.csv')
write_csv_to_s3(df_val, BUCKET_NAME, 'features/val.csv')

log.info('CLV rebuild complete. train: %d rows, val: %d rows', len(df_train), len(df_val))